# EDA on Olympic data


## Create Data Frame (Exercise 0:0 a)
- Read as CSV using Spark
- Display using display()

In [0]:
%python
DATA_PATH = '/Volumes/olympics/bronze/raw_data'
ATHLETE_PATH = DATA_PATH + '/athlete_events.csv'
NOC_PATH = DATA_PATH + '/noc_regions.csv'

df_olympics = spark.read.csv(ATHLETE_PATH, header=True)
df_noc = spark.read.csv(NOC_PATH, header=True)

In [0]:
display(df_olympics)

In [0]:
display(df_noc)

### Use spark columns method to find out the columns (Exercise 0:0 b)

In [0]:
df_olympics.columns

## Schema for data frame
Since all is String we import a schema to determin columntype
- Check in frame above to se what it should be.
- Use StructType to create shema
- Create new df with schema

In [0]:
%python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, ShortType, ByteType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", FloatType()),
  StructField("Weight", FloatType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])

df_olympics_schema = spark.read.csv(ATHLETE_PATH, header=True, schema=schema)

df_olympics_schema.display()

## Control null-vales
- Check all null-values and sum them up

#### Functions explained
- df_olympics_schema.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_olympics_schema.columns])

| Function | Does | 
|----------|------| 
|col()|Makes it possible to run funktions|
|isNull()|Returns True if Null else False|
|cast("int")|Translate True to 1 False to 0|
|sum()|Sum the value of every column|
|alias()|Change header-title|


In [0]:
%python
from pyspark.sql.functions import col, sum

nulls = df_olympics_schema.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_olympics_schema.columns])

display(nulls)

### Swedish Olympians

In [0]:
df_olympic_sweds = df_olympics_schema.filter("NOC = 'SWE'")

df_olympic_sweds.display()

### Most olympians in the north

In [0]:
df_olympic_most_north = df_olympics_schema.groupBy("NOC").count().filter("NOC IN ('SWE','NOR','FIN','DEN','ISL')")

df_olympic_most_north.createOrReplaceTempView("df_olympic_most_north")        
                                                                       
df_olympic_most_north.display()                                                                    

### Swedish Olympian medalists

In [0]:
df_olympics_schema.createOrReplaceTempView("df_olympics_schema")

df_swe_medalist = spark.sql("""
                          SELECT
                            Name,
                            age,
                            weight,
                            height,
                            year,
                            sport,
                            medal
                          FROM df_olympics_schema
                          WHERE
                            NOC = 'SWE' AND medal IN ('Gold','Silver','Bronze')
                          """)
df_swe_medalist.createOrReplaceTempView("df_swe_medalist")                          
df_swe_medalist.display()

In [0]:
df_noc.createOrReplaceTempView("df_noc")

### Numbers of medals to Sweden

In [0]:
df_swe_medals = spark.sql("""
                          SELECT
                            sport,
                            count(medal) as medals
                          FROM df_olympics_schema
                          WHERE
                            NOC = 'SWE' AND medal IN ('Gold','Silver','Bronze')
                          GROUP BY
                            sport
                          ORDER BY
                            medals DESC
                          """)
df_swe_medals.createOrReplaceTempView("df_swe_medals")                        
df_swe_medals.display()

In [0]:
fig = df_swe_medals.plot(kind="bar", y="sport", x="medals", title="Swedish Medals")

fig.update_layout(yaxis= {'autorange': 'reversed'})
fig.show()

## With SQL

In [0]:
%sql
FROM df_olympics_schema LIMIT 10;

In [0]:
%sql

SHOW CATALOGS;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS olympics;

-- CREATE VOLUME IF NOT EXISTS olympics.raw_data;

CREATE SCHEMA IF NOT EXISTS olympics.bronze;
CREATE SCHEMA IF NOT EXISTS olympics.silver;
CREATE SCHEMA IF NOT EXISTS olympics.gold;

CREATE TABLE IF NOT EXISTS olympics.bronze.olympics
SELECT * FROM df_olympics_schema;

CREATE TABLE IF NOT EXISTS olympics.bronze.noc_country
SELECT * FROM df_noc;

CREATE TABLE IF NOT EXISTS olympics.silver.sweden_medals
SELECT * FROM df_swe_medals;

CREATE TABLE IF NOT EXISTS olympics.silver.sweden_medalist
SELECT * FROM df_swe_medalist;

CREATE TABLE IF NOT EXISTS olympics.silver.north_olympians
SELECT * FROM df_olympic_most_north;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS olympics.bronze.raw_data;

### The 10 oldest atheletes, their age and the sport (exercise 0:0 c)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS olympics.silver.oledest_olympians AS(
    SELECT 
        name,
        Age,
        sport,
        count(name) as Nr_Of_Times,
        SUM(CASE WHEN `Medal` IN ('Gold', 'Silver', 'Bronze') THEN 1 ELSE 0 END) AS Total_Medals
    FROM olympics.bronze.olympics
    WHERE Age is not null
    GROUP BY name, Age, sport
    ORDER BY Age DESC
    LIMIT 10
);

SELECT * FROM olympics.silver.oledest_olympians;



### The 10 youngest atheletes, their age and the sport (exercise 0:0 d)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS olympics.silver.youngest_olympians AS(
    SELECT 
        name,
        Age,
        sport,
        count(name) as Nr_Of_Times,
        SUM(CASE WHEN `Medal` IN ('Gold', 'Silver', 'Bronze') THEN 1 ELSE 0 END) AS Total_Medals
    FROM olympics.bronze.olympics
    WHERE Age is not null
    GROUP BY name, Age, sport
    ORDER BY Age ASC
    LIMIT 10
);

SELECT * FROM olympics.silver.youngest_olympians;



### The 5 sports with highest median age (exercise 0:0 e)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS olympics.silver.highest_median_age AS(
    SELECT
        Sport,
        median(Age) AS Median_Age
    FROM olympics.bronze.olympics
    GROUP BY Sport
    ORDER BY median_age DESC
    LIMIT 5
);

SELECT * FROM olympics.silver.highest_median_age;

### The 5 sports with lowest  median age (exercise 0:0 f)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS olympics.silver.lowest_median_age AS(
    SELECT
        Sport,
        median(Age) AS Median_Age
    FROM olympics.bronze.olympics
    WHERE Age is not null
    GROUP BY Sport
    ORDER BY median_age ASC
    LIMIT 5
);

SELECT * FROM olympics.silver.lowest_median_age;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS olympics.silver.gold_medal_countries AS(
    SELECT
        count(o.medal) AS medals,
        o.NOC,
        n.region AS Country
    FROM olympics.bronze.olympics o
    LEFT JOIN olympics.bronze.noc_country n ON o.NOC = n.NOC
    WHERE Medal = 'Gold'
    GROUP BY
        o.NOC,
        Country
    ORDER BY
        medals DESC
    LIMIT 10);

SELECT * FROM olympics.silver.gold_medal_countries;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS olympics.silver.medal_countries AS(
    SELECT
        count(o.medal) AS medals,
        o.NOC,
        n.region AS Country
    FROM olympics.bronze.olympics o
    LEFT JOIN olympics.bronze.noc_country n ON o.NOC = n.NOC
    WHERE o.medal IN('Gold', 'Silver', 'Bronze')
    GROUP BY
        o.NOC,
        Country
    ORDER BY
        medals DESC
    LIMIT 10);

SELECT * FROM olympics.silver.medal_countries;

In [0]:
df_male_female = spark.sql("""
          SELECT
            year,
            COUNT_IF(Sex = 'M') AS Male,
            COUNT_IF(Sex = 'F') AS Female
          FROM olympics.bronze.olympics
          WHERE Year is NOT NULL
          GROUP BY Year
          ORDER BY Year ASC;
""")

df_male_female.display()

In [0]:
df_male_female.plot(kind='line', x='year', y=['Male', 'Female'])